In [ ]:
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor
from sklearn.ensemble import ExtraTreesRegressor

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# ============================================================
# 1. LOW-LEVEL FEATURE ENGINEERING FROM JSON PROFILES
# ============================================================

CONDITION_CODES = ["AT", "HT", "HD", "DB"]
CONDITIONS_COL = "Conditions"
OUTPATIENT_COL = "Out patient costs"  # <-- change here if column name differs


def _safe_parse_obj(obj):
    """
    Handle dicts / JSON strings / NaNs for 'Conditions' and 'Out patient costs'.
    Returns a dict.
    """
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, str):
        obj = obj.strip()
        if not obj:
            return {}
        try:
            return json.loads(obj)
        except Exception:
            # Fallback for strings like "AT:2,HT:3"
            result = {}
            for part in obj.replace("{", "").replace("}", "").split(","):
                if ":" in part:
                    k, v = part.split(":")
                    k = k.strip().strip('"').strip("'")
                    try:
                        v = float(v)
                    except Exception:
                        v = 0.0
                    result[k] = v
            return result
    return {}


def engineer_base_features(df_profiles: pd.DataFrame) -> pd.DataFrame:
    """
    Take the raw JSON profiles (train or test) and create tabular features:
    - Demographics
    - Per-condition severity
    - Aggregated severity stats
    - Outpatient cost statistics
    - A hand-crafted risk_score to be used later
    """
    df = df_profiles.copy()

    # --- Sex encoding ---
    if "Sex" in df.columns:
        df["sex_male"] = (df["Sex"] == "M").astype(int)
        df["sex_female"] = (df["Sex"] == "F").astype(int)
    else:
        df["sex_male"] = 0
        df["sex_female"] = 0

    # --- Age features ---
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["age_squared"] = df["Age"] ** 2
    df["age_minus_65"] = df["Age"] - 65
    df["age_over_75"] = (df["Age"] > 75).astype(int)
    df["age_over_80"] = (df["Age"] > 80).astype(int)

    # --- Parse chronic conditions ---
    cond_dicts = df[CONDITIONS_COL].apply(_safe_parse_obj)

    for code in CONDITION_CODES:
        df[f"{code}_severity"] = cond_dicts.apply(lambda d: float(d.get(code, 0.0)))
        df[f"{code}_present"] = (df[f"{code}_severity"] > 0).astype(int)

    # Aggregated severity stats
    severity_cols = [f"{c}_severity" for c in CONDITION_CODES]

    df["total_severity"] = df[severity_cols].sum(axis=1)
    df["max_severity"] = df[severity_cols].max(axis=1)
    df["min_severity"] = df[severity_cols].min(axis=1)
    df["avg_severity"] = df["total_severity"] / (
        df[severity_cols].gt(0).sum(axis=1).replace(0, 1)
    )
    df["total_conditions"] = df[severity_cols].gt(0).sum(axis=1)

    # Simple comorbidity counts
    df["num_two_plus_conditions"] = (df["total_conditions"] >= 2).astype(int)
    df["num_three_plus_conditions"] = (df["total_conditions"] >= 3).astype(int)

    # --- Parse outpatient costs ---
    if OUTPATIENT_COL in df.columns:
        cost_dicts = df[OUTPATIENT_COL].apply(_safe_parse_obj)
    else:
        cost_dicts = pd.Series([{}] * len(df), index=df.index)

    def _cost_stats(d):
        if not d:
            return {
                "op_years": 0,
                "op_total_cost": 0.0,
                "op_mean_cost": 0.0,
                "op_std_cost": 0.0,
                "op_min_cost": 0.0,
                "op_max_cost": 0.0,
                "op_trend": 0.0,
                "op_recent_cost": 0.0,
                "op_first_cost": 0.0,
            }
        years = sorted(d.keys())
        vals = np.array([float(d[y]) for y in years])
        op_total = float(vals.sum())
        op_mean = float(vals.mean())
        op_std = float(vals.std()) if len(vals) > 1 else 0.0
        op_min = float(vals.min())
        op_max = float(vals.max())
        op_first = float(vals[0])
        op_recent = float(vals[-1])
        op_trend = float(op_recent - op_first)
        return {
            "op_years": len(years),
            "op_total_cost": op_total,
            "op_mean_cost": op_mean,
            "op_std_cost": op_std,
            "op_min_cost": op_min,
            "op_max_cost": op_max,
            "op_trend": op_trend,
            "op_recent_cost": op_recent,
            "op_first_cost": op_first,
        }

    cost_features = cost_dicts.apply(_cost_stats).apply(pd.Series)
    df = pd.concat([df, cost_features], axis=1)

    # Derived utilization features
    df["op_cost_per_year"] = df["op_total_cost"] / df["op_years"].replace(0, 1)
    df["op_recent_ratio"] = df["op_recent_cost"] / df["op_total_cost"].replace(0, 1)
    df["op_trend_per_year"] = df["op_trend"] / df["op_years"].replace(0, 1)

    # --- Hand-crafted risk score (used later) ---
    df["risk_score"] = (
        1.2 * df["AT_severity"]
        + 1.5 * df["HT_severity"]
        + 2.0 * df["HD_severity"]
        + 1.8 * df["DB_severity"]
        + 0.1 * df["age_minus_65"].clip(lower=0)
        + 0.05
        * df["op_total_cost"].clip(upper=df["op_total_cost"].quantile(0.99))
    )

    return df


# ============================================================
# 2. HIGH-LEVEL "SMART" FEATURES (ADAPTED FROM YOUR SCRIPT)
# ============================================================

def create_smart_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create high-value interaction features ON TOP of the base features
    that engineer_base_features already created.
    """
    df = df.copy()

    # Core interactions with age
    df["age_severity"] = df["Age"] * df["total_severity"]
    df["age_conditions"] = df["Age"] * df["total_conditions"]
    df["age_squared_severity"] = df["age_squared"] * df["total_severity"]

    df["severity_per_condition"] = df["total_severity"] / (df["total_conditions"] + 1)
    df["severity_concentration"] = df["max_severity"] / (df["avg_severity"] + 1e-2)

    # Individual condition risk interactions
    for cond in CONDITION_CODES:
        df[f"{cond}_age_risk"] = df[f"{cond}_severity"] * df["Age"]
        df[f"{cond}_squared"] = df[f"{cond}_severity"] ** 2

    # Comorbidity patterns
    df["AT_HT_risk"] = df["AT_severity"] * df["HT_severity"]
    df["HT_HD_risk"] = df["HT_severity"] * df["HD_severity"]
    df["HD_DB_risk"] = df["HD_severity"] * df["DB_severity"]
    df["AT_DB_risk"] = df["AT_severity"] * df["DB_severity"]

    df["cardio_risk"] = df["AT_severity"] * df["HT_severity"] * df["HD_severity"]
    df["metabolic_risk"] = df["HT_severity"] * df["DB_severity"] * df["Age"]

    severity_cols = [f"{c}_severity" for c in CONDITION_CODES]
    df["severity_std"] = df[severity_cols].std(axis=1)
    df["severity_range"] = df[severity_cols].max(axis=1) - df[severity_cols].min(axis=1)
    df["severity_cv"] = df["severity_std"] / (df["avg_severity"] + 1e-2)

    # Risk flag features
    df["high_risk"] = ((df["Age"] > 78) & (df["total_severity"] > 8)).astype(int)
    df["extreme_age"] = (df["Age"] > 80).astype(int)
    df["high_severity"] = (df["total_severity"] > 10).astype(int)
    df["multi_severe"] = (df["total_conditions"] >= 3).astype(int)

    df["composite_risk"] = (
        df["Age"] * 0.3
        + df["total_severity"] * 2.0
        + df["total_conditions"] * 5.0
        + df["max_severity"] * 3.0
        + df["op_total_cost"] * 0.01
    )

    # Log & power transforms
    df["log_age"] = np.log(df["Age"].clip(lower=1))
    df["log_severity"] = np.log1p(df["total_severity"])
    df["log_risk"] = np.log1p(df["risk_score"].clip(lower=0))
    df["sqrt_age"] = np.sqrt(df["Age"])
    df["sqrt_severity"] = np.sqrt(df["total_severity"])
    df["log_op_total"] = np.log1p(df["op_total_cost"])
    df["log_op_recent"] = np.log1p(df["op_recent_cost"])

    # Age groups / severity groups
    df["age_group"] = pd.cut(
        df["Age"], bins=[0, 70, 75, 80, 120], labels=[0, 1, 2, 3]
    ).astype(int)
    df["severity_group"] = pd.cut(
        df["total_severity"], bins=[0, 4, 7, 10, 100], labels=[0, 1, 2, 3]
    ).astype(int)

    # Outpatient utilization vs severity
    df["cost_per_severity"] = df["op_total_cost"] / (df["total_severity"] + 1)
    df["recent_cost_per_severity"] = df["op_recent_cost"] / (df["total_severity"] + 1)

    return df


# ============================================================
# 3. HYPERPARAMETER OPTIMIZATION (OPTIONAL)
# ============================================================

def optimize_lgbm(X_train, y_train, n_trials=30):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 1500, 4500),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.005, 0.05, log=True
            ),
            "num_leaves": trial.suggest_int("num_leaves", 24, 80),
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
            "objective": "mae",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "verbose": -1,
        }
        model = LGBMRegressor(**params)

        kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for tr_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            scores.append(mean_absolute_error(y_val, preds))
        return float(np.mean(scores))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params


def optimize_xgb(X_train, y_train, n_trials=30):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 1500, 4500),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.005, 0.05, log=True
            ),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
            "gamma": trial.suggest_float("gamma", 0.0, 1.0),
            "objective": "reg:absoluteerror",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "tree_method": "hist",
        }
        model = XGBRegressor(**params)

        kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for tr_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            preds = model.predict(X_val)
            scores.append(mean_absolute_error(y_val, preds))
        return float(np.mean(scores))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params


def optimize_catboost(X_train, y_train, n_trials=30):
    def objective(trial):
        params = {
            "iterations": trial.suggest_int("iterations", 1500, 4500),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.005, 0.05, log=True
            ),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.6, 1.0),
            "loss_function": "MAE",
            "random_seed": RANDOM_STATE,
            "verbose": 0,
        }
        model = CatBoostRegressor(**params)

        kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for tr_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
            preds = model.predict(X_val)
            scores.append(mean_absolute_error(y_val, preds))
        return float(np.mean(scores))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params


# ============================================================
# 4. OUT-OF-FOLD PREDICTIONS
# ============================================================

def get_oof_predictions(model, X, y, X_test, n_folds=10, model_name="model"):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        oof_preds[val_idx] = model.predict(X_val)
        fold_mae = mean_absolute_error(y_val, oof_preds[val_idx])
        print(f"[{model_name}] Fold {fold+1}/{n_folds} MAE: {fold_mae:.3f}")

        test_preds += model.predict(X_test) / n_folds

    overall_mae = mean_absolute_error(y, oof_preds)
    print(f"[{model_name}] OOF MAE: {overall_mae:.3f}")
    return oof_preds, test_preds, overall_mae


# ============================================================
# 5. MAIN PIPELINE
# ============================================================

def main(
    train_csv_path="train.csv",
    train_json_path="train.json",
    test_json_path="test.json",
    sample_sub_path="sample_submission.csv",
    do_optuna=True,
    n_trials=30,
):
    TARGET = "TotalCosts"
    ID_COL = "PatientID"

    print("=" * 70)
    print("ADVANCED HEALTHCARE CLAIMS – JSON PROFILES + STACKED ENSEMBLES")
    print("=" * 70)

    # ---------- Load base files ----------
    train_targets = pd.read_csv(train_csv_path)
    train_profiles = pd.read_json(train_json_path)
    test_profiles = pd.read_json(test_json_path)
    sample_sub = pd.read_csv(sample_sub_path)

    assert TARGET in train_targets.columns, f"{TARGET} not found in train.csv"
    assert ID_COL in train_targets.columns, f"{ID_COL} not found in train.csv"

    # Join targets with profiles
    train = pd.merge(train_targets, train_profiles, on=ID_COL, how="left")
    test = pd.merge(sample_sub[[ID_COL]], test_profiles, on=ID_COL, how="left")

    print(f"Train shape (raw): {train.shape}, Test shape (raw): {test.shape}")

    # ---------- Feature engineering ----------
    train_fe = engineer_base_features(train)
    test_fe = engineer_base_features(test)

    train_fe = create_smart_features(train_fe)
    test_fe = create_smart_features(test_fe)

    print(f"Train shape (engineered): {train_fe.shape}, Test shape (engineered): {test_fe.shape}")

    # Drop non-numeric / leakage columns
    drop_cols = [TARGET, ID_COL, "Sex", CONDITIONS_COL, OUTPATIENT_COL]
    feature_cols = [c for c in train_fe.columns if c not in drop_cols]

    X = train_fe[feature_cols].fillna(0.0)
    y = train_fe[TARGET]
    X_test = test_fe[feature_cols].fillna(0.0)

    print(f"Feature count: {X.shape[1]}")
    print(f"Train samples: {X.shape[0]}, Test samples: {X_test.shape[0]}")

    # ---------- Scaling ----------
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    # ---------- Hyperparameter optimization ----------
    if do_optuna:
        print("\n" + "=" * 70)
        print("STAGE 1: HYPERPARAMETER OPTIMIZATION")
        print("=" * 70)

        print("\n[1/3] Optimizing LightGBM...")
        lgbm_params = optimize_lgbm(X_scaled, y, n_trials=n_trials)
        print("Best LightGBM params:", lgbm_params)

        print("\n[2/3] Optimizing XGBoost...")
        xgb_params = optimize_xgb(X_scaled, y, n_trials=n_trials)
        print("Best XGBoost params:", xgb_params)

        print("\n[3/3] Optimizing CatBoost...")
        cat_params = optimize_catboost(X_scaled, y, n_trials=n_trials)
        print("Best CatBoost params:", cat_params)
    else:
        # Paste your best params here once you've found them
        lgbm_params = dict(
            n_estimators=3000,
            learning_rate=0.01,
            num_leaves=31,
            max_depth=7,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=1.0,
            objective="mae",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
        xgb_params = dict(
            n_estimators=3000,
            learning_rate=0.02,
            max_depth=8,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            gamma=0.1,
            objective="reg:absoluteerror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
        )
        cat_params = dict(
            iterations=3000,
            learning_rate=0.02,
            depth=6,
            l2_leaf_reg=3.0,
            subsample=0.8,
            colsample_bylevel=0.8,
            loss_function="MAE",
            random_seed=RANDOM_STATE,
            verbose=0,
        )

    print("\n" + "=" * 70)
    print("STAGE 2: BASE MODELS WITH OOF PREDICTIONS")
    print("=" * 70)

    # LightGBM (optimized/default)
    lgbm = LGBMRegressor(**lgbm_params)
    oof_lgbm, test_lgbm, mae_lgbm = get_oof_predictions(
        lgbm, X_scaled, y, X_test_scaled, n_folds=10, model_name="LightGBM"
    )

    # A slightly different LightGBM variant (more conservative)
    lgbm2 = LGBMRegressor(
        n_estimators=3500,
        learning_rate=0.008,
        num_leaves=40,
        max_depth=8,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=1.5,
        objective="mae",
        random_state=RANDOM_STATE + 1,
        n_jobs=-1,
        verbose=-1,
    )
    oof_lgbm2, test_lgbm2, mae_lgbm2 = get_oof_predictions(
        lgbm2, X_scaled, y, X_test_scaled, n_folds=10, model_name="LightGBM_v2"
    )

    # XGBoost
    xgb = XGBRegressor(**xgb_params)
    oof_xgb, test_xgb, mae_xgb = get_oof_predictions(
        xgb, X_scaled, y, X_test_scaled, n_folds=10, model_name="XGBoost"
    )

    # CatBoost
    cat = CatBoostRegressor(**cat_params)
    oof_cat, test_cat, mae_cat = get_oof_predictions(
        cat, X_scaled, y, X_test_scaled, n_folds=10, model_name="CatBoost"
    )

    # ExtraTrees
    et = ExtraTreesRegressor(
        n_estimators=700,
        max_depth=22,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    oof_et, test_et, mae_et = get_oof_predictions(
        et, X_scaled, y, X_test_scaled, n_folds=10, model_name="ExtraTrees"
    )

    # Huber Regressor
    huber = HuberRegressor(epsilon=1.35, alpha=0.0005, max_iter=1000)
    oof_huber, test_huber, mae_huber = get_oof_predictions(
        huber, X_scaled, y, X_test_scaled, n_folds=10, model_name="Huber"
    )

    print("\n" + "=" * 70)
    print("STAGE 3: STACKING / BLENDING")
    print("=" * 70)

    oof_df = pd.DataFrame(
        {
            "lgbm": oof_lgbm,
            "lgbm2": oof_lgbm2,
            "xgb": oof_xgb,
            "cat": oof_cat,
            "et": oof_et,
            "huber": oof_huber,
        }
    )
    test_df = pd.DataFrame(
        {
            "lgbm": test_lgbm,
            "lgbm2": test_lgbm2,
            "xgb": test_xgb,
            "cat": test_cat,
            "et": test_et,
            "huber": test_huber,
        }
    )

    meta_models = {
        "ridge": Ridge(alpha=5.0),
        "elastic": ElasticNet(alpha=0.5, l1_ratio=0.3, max_iter=5000),
        "huber_meta": HuberRegressor(epsilon=1.2, alpha=0.01),
    }

    best_mae = float("inf")
    best_name = None
    best_pred = None
    test_ids = test[ID_COL].values

    for name, meta in meta_models.items():
        print(f"\nMeta-model: {name}")
        oof_meta = np.zeros(len(oof_df))
        test_meta = np.zeros(len(test_df))

        kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
        for fold, (tr_idx, val_idx) in enumerate(kf.split(oof_df)):
            X_tr, X_val = oof_df.iloc[tr_idx], oof_df.iloc[val_idx]
            y_tr = y.iloc[tr_idx]

            meta.fit(X_tr, y_tr)
            oof_meta[val_idx] = meta.predict(X_val)
            test_meta += meta.predict(test_df) / 10

        mae_meta = mean_absolute_error(y, oof_meta)
        print(f"  -> {name} OOF MAE: {mae_meta:.3f}")

        sub_name = f"submission_meta_{name}.csv"
        pd.DataFrame({ID_COL: test_ids, TARGET: test_meta}).to_csv(
            sub_name, index=False
        )
        print(f"  -> Saved {sub_name}")

        if mae_meta < best_mae:
            best_mae = mae_meta
            best_name = f"meta_{name}"
            best_pred = test_meta

    print("\nSimple weighted average blend...")
    weights = np.array([0.25, 0.20, 0.25, 0.20, 0.05, 0.05])
    oof_avg = (
        oof_lgbm * weights[0]
        + oof_lgbm2 * weights[1]
        + oof_xgb * weights[2]
        + oof_cat * weights[3]
        + oof_et * weights[4]
        + oof_huber * weights[5]
    )
    test_avg = (
        test_lgbm * weights[0]
        + test_lgbm2 * weights[1]
        + test_xgb * weights[2]
        + test_cat * weights[3]
        + test_et * weights[4]
        + test_huber * weights[5]
    )
    mae_avg = mean_absolute_error(y, oof_avg)
    print(f"  -> Weighted average OOF MAE: {mae_avg:.3f}")

    pd.DataFrame({ID_COL: test_ids, TARGET: test_avg}).to_csv(
        "submission_weighted_avg.csv", index=False
    )
    print("  -> Saved submission_weighted_avg.csv")

    if mae_avg < best_mae:
        best_mae = mae_avg
        best_name = "weighted_avg"
        best_pred = test_avg

    # Final best
    final_sub_name = "submission_BEST_ultimate.csv"
    pd.DataFrame({ID_COL: test_ids, TARGET: best_pred}).to_csv(
        final_sub_name, index=False
    )

    print("\n" + "=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)
    print(f"LightGBM:       {mae_lgbm:.3f}")
    print(f"LightGBM_v2:    {mae_lgbm2:.3f}")
    print(f"XGBoost:        {mae_xgb:.3f}")
    print(f"CatBoost:       {mae_cat:.3f}")
    print(f"ExtraTrees:     {mae_et:.3f}")
    print(f"Huber:          {mae_huber:.3f}")
    print(f"Weighted avg:   {mae_avg:.3f}")
    print(f"\nBEST ensemble:  {best_name} with OOF MAE = {best_mae:.3f}")
    print(f"Final submission -> {final_sub_name}")


if __name__ == "__main__":
    # Flip do_optuna to False for fast local iterations.
    main(
        train_csv_path="train.csv",
        train_json_path="train.json",
        test_json_path="test.json",
        sample_sub_path="sample_submission.csv",
        do_optuna=True,
        n_trials=30,
    )